# Práctica 2 · Tautomería y protonación como microespacio químico
### Enumeración de tautómeros, GFN2-xTB y distribución de Boltzmann

> **Bloque 1 — Generación de estructuras**  ·  Manual de Química Computacional UNAM

| | |
|:--|:--|
| **Molécula semilla** | 2-piridona / 2-hidroxipiridina — `O=c1ccccn1H` |
| **Herramientas** | RDKit · xtb · py3Dmol |
| **Tiempo estimado** | 1.5 h (semilla) + 2 h (bosque + análisis) |
| **Requisitos previos** | Práctica 1 · Equilibrio químico · Energía libre de Gibbs |

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/qcmanual/del-orbital-al-espacio-quimico/blob/main/notebooks/p02.ipynb)


---
## ⚙️ Configuración del entorno

**Ejecuta esta celda primero.** Reutiliza el entorno `qc-manual` de la Práctica 1.


In [ ]:
# ── Instalación (solo si estás en Google Colab) ───────────────
# !pip install rdkit py3Dmol -q

import warnings, subprocess, shutil, os, re, time, pathlib
warnings.filterwarnings('ignore')
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib; matplotlib.rcParams['figure.dpi'] = 120
from IPython.display import display, HTML, Image

# Verificar RDKit
try:
    from rdkit import Chem, __version__ as rv
    from rdkit.Chem import AllChem, Draw, rdMolDescriptors, rdmolfiles
    from rdkit.Chem.Draw import rdMolDraw2D, MolsToGridImage
    from rdkit.Chem.MolStandardize import rdMolStandardize
    print(f'\u2713  RDKit {rv}')
except ImportError:
    print('\u2717  RDKit no encontrado')

# Verificar xtb
if shutil.which('xtb'):
    v = subprocess.run(['xtb','--version'], capture_output=True, text=True)
    print(f'\u2713  {v.stdout.strip().splitlines()[0]}')
else:
    print('\u26a0  xtb no encontrado en PATH')
    print('   Instala con: conda install -c conda-forge xtb')

# Verificar py3Dmol
try:
    import py3Dmol; PY3DMOL = True
    print('\u2713  py3Dmol disponible')
except ImportError:
    PY3DMOL = False
    print('\u26a0  py3Dmol no instalado  ->  pip install py3Dmol')

# Constantes termodinámicas
R = 8.314e-3   # kJ/(mol·K)
T_K = 298.15   # K

print('\n\u2500\u2500\u2500 Entorno listo \u2713 \u2500\u2500\u2500')


---
## 📝 Preguntas previas

Escribe tus respuestas **antes de ejecutar cualquier código**.
Al final de la práctica volverás a leerlas para ver cómo cambió tu razonamiento.


**Pregunta 1 (conceptual).** La guanina tiene cuatro tautómeros
relevantes biológicamente: la forma ceto-amino (dominante en ADN),
la enol-amino, la ceto-imino y la enol-imino.
Sin calcular nada, ¿cuál esperas que sea el más estable y por qué?
¿Qué consecuencia tendría para el apareamiento de bases si
el tautómero minoritario estuviera presente en el 0.01 % de las moléculas?


In [ ]:
# ── Escribe tu respuesta a la Pregunta 1 aquí ─────────────────
respuesta_1 = (
    'El tautómero más estable de la guanina es...\n'
    'porque...\n'
    'La consecuencia para el apareamiento de bases sería...'
)
print('Respuesta 1 guardada.')


**Pregunta 2 (predictivo).** Para la 2-piridona hay dos formas:
la forma **ceto** (lactama: C=O y N-H) y la forma **enol**
(2-hidroxipiridina: C-OH, N sin H).
¿Cuál esperas que sea más estable en fase gas?
¿Cambiaría ese orden en agua? ¿Por qué?


In [ ]:
# ── Escribe tu respuesta a la Pregunta 2 aquí ─────────────────
respuesta_2 = (
    'En fase gas espero que sea más estable la forma...\n'
    'porque...\n'
    'En agua cambiaría / no cambiaría porque...'
)
print('Respuesta 2 guardada.')


**Pregunta 3 (crítico).** El pipeline de esta práctica calcula
energías en vacío y aplica la distribución de Boltzmann.
¿En qué circunstancias daría predicciones incorrectas sobre
qué tautómero predomina en solución acuosa?
¿Qué habría que añadir al pipeline para corregirlo?


In [ ]:
# ── Escribe tu respuesta a la Pregunta 3 aquí ─────────────────
respuesta_3 = (
    'El protocolo en vacío fallaría cuando...\n'
    'Para corregirlo habría que añadir...'
)
print('Respuesta 3 guardada.')


---
## Introducción

Cuando escribes el SMILES de una molécula, implícitamente eliges
una forma tautómera y un estado de protonación. En condiciones
reales — en solución, en el sitio activo de una enzima — la misma
molécula puede existir en múltiples formas que se interconvierten
rápidamente. Esta **especiación química** tiene consecuencias
directas sobre la reactividad, la solubilidad y la afinidad
de unión a proteínas. Si el cálculo se ejecuta sobre el tautómero
incorrecto, todos los resultados subsecuentes son los de una
especie que puede ser minoritaria o inexistente.

### Pipeline de esta práctica

```
SMILES  ──[TautomerEnumerator]──▶  Lista de tautómeros (2D)
   ↓
Cada tautómero ──[ETKDG + MMFF94]──▶  Geometría 3D inicial
   ↓
Geometría ──[xtb --opt tight --ohess]──▶  G(298 K) por forma
   ↓
G_i ──[Boltzmann: p_i = exp(-G_i/RT)/Σ]──▶  Poblaciones de equilibrio
   ↓
Resultados ──[pandas + matplotlib]──▶  Dataset de 40 moléculas
```

> 📖 **Antes de continuar:** lee la Sección 2.1 del manual
> (Tautomería, protómeros y distribución de Boltzmann).


---
## 🌱 Semilla · Paso 1 — Enumeración de tautómeros con RDKit

### ¿Qué hace este paso?

`TautomerEnumerator` aplica transformaciones SMARTS predefinidas
(reglas de Sitzmann) para generar todas las formas tautómeras posibles.
El procedimiento: (i) identifica patrones reactivos (ceto-enol,
imino-amino), (ii) aplica las transformaciones de forma combinatoria,
(iii) elimina duplicados canónicos.

### Instrucciones

1. Ejecuta la celda.
2. Observa cuántos tautómeros encuentra RDKit para la 2-piridona.
3. **Mira la rejilla de estructuras**: identifica la forma ceto
   (tiene C=O explícito y N-H) y la forma enol (tiene C-OH).
4. ¿Hay alguna forma que te parezca químicamente improbable?
   Anota cuál y por qué.


In [ ]:
# ── Paso 1: SMILES → tautómeros ───────────────────────────────
SMILES_ENTRADA = 'O=c1ccccn1H'   # 2-piridona (forma ceto)
NOMBRE         = '2-piridona'

mol = Chem.MolFromSmiles(SMILES_ENTRADA)
assert mol is not None, f'SMILES inválido: {SMILES_ENTRADA}'

print(f'Molécula de entrada : {NOMBRE}')
print(f'SMILES              : {SMILES_ENTRADA}')
print(f'Fórmula             : {rdMolDescriptors.CalcMolFormula(mol)}')
print(f'Átomos pesados      : {mol.GetNumHeavyAtoms()}')
print()

# Configurar enumerador
enumerador = rdMolStandardize.TautomerEnumerator()
enumerador.SetMaxTautomers(100)
enumerador.SetRemoveSp3Stereo(True)

tautomeros = enumerador.Enumerate(mol)
n_tau      = len(tautomeros)

print(f'Tautómeros encontrados: {n_tau}')
print()
print('SMILES de cada tautómero:')
for i, tau in enumerate(tautomeros):
    print(f'  T{i+1:02d}: {Chem.MolToSmiles(tau)}')


In [ ]:
# ── Rejilla 2D de tautómeros ───────────────────────────────────
# Los índices T01, T02... se usarán para identificar archivos
# en todos los pasos siguientes.

img = MolsToGridImage(
    list(tautomeros),
    molsPerRow=4,
    subImgSize=(280, 220),
    legends=[f'T{i+1:02d}' for i in range(n_tau)]
)
img.save(f'{NOMBRE}_tautomeros_2D.png')
display(img)

print(f'Imagen guardada: {NOMBRE}_tautomeros_2D.png')
print()
print('Anota en tu cuaderno:')
print('  · ¿Cuál tautómero es la forma CETO? (C=O + N-H)')
print('  · ¿Cuál es la forma ENOL? (C-OH, N sin H)')
print('  · ¿Alguna forma te parece químicamente inverosímil?')


---
## Paso 2 — Geometría 3D para cada tautómero (ETKDG + MMFF94)

### ¿Qué hace este paso?

Para cada tautómero repetimos el pipeline de la Práctica 1:
ETKDG genera coordenadas 3D iniciales, MMFF94 las limpia de
tensiones geométricas. Cada forma tautómera tiene conectividad
distinta (el H migratorio está en un lugar diferente), por lo que
cada una necesita su propia geometría.

### Instrucciones

1. Ejecuta la celda. Genera un archivo `_FF.xyz` por tautómero.
2. Observa si algún tautómero **falla** en la incrustación.
   ¿Por qué crees que falla?
3. Compara las energías MMFF94 de los distintos tautómeros:
   ¿ya hay diferencias en este nivel? ¿Tiene sentido?


In [ ]:
# ── Paso 2: geometría 3D para cada tautómero ──────────────────
directorio = f'{NOMBRE}_tautomeros'
os.makedirs(directorio, exist_ok=True)

registros_geom = []

for i, tau in enumerate(tautomeros):
    etiqueta = f'{NOMBRE}_T{i+1:02d}'

    # Añadir H y embeber
    tau_h  = Chem.AddHs(tau)
    params = AllChem.ETKDGv3()
    params.randomSeed = 42
    ok = AllChem.EmbedMolecule(tau_h, params)

    if ok == -1:
        print(f'  T{i+1:02d}: ✗ fallo en incrustación')
        registros_geom.append({'etiqueta': etiqueta, 'smiles': Chem.MolToSmiles(tau),
                                'embed_ok': False, 'e_ff_kcalmol': None})
        continue

    # Pre-optimización MMFF94
    props = AllChem.MMFFGetMoleculeProperties(tau_h)
    ff    = AllChem.MMFFGetMoleculeForceField(tau_h, props)
    if ff is None:
        ff = AllChem.UFFGetMoleculeForceField(tau_h)
    ff.Minimize(maxIts=2000)
    e_ff = ff.CalcEnergy()

    xyz_path = os.path.join(directorio, f'{etiqueta}_FF.xyz')
    rdmolfiles.MolToXYZFile(tau_h, xyz_path)

    registros_geom.append({'etiqueta': etiqueta, 'smiles': Chem.MolToSmiles(tau),
                            'embed_ok': True, 'e_ff_kcalmol': e_ff})
    print(f'  T{i+1:02d}: ✓  E_MMFF94 = {e_ff:.2f} kcal/mol')

df_geom = pd.DataFrame(registros_geom)
n_ok = df_geom['embed_ok'].sum()
print(f'\nTotal: {n_ok}/{n_tau} tautómeros con geometría lista para xTB')
print()
print('Observa las energías MMFF94:')
print('  ¿Ya se ordenan de la misma forma que esperas al final?')
print('  ¿Qué limitaciones tiene MMFF94 para comparar tautómeros?')


---
## Paso 3 — Optimización + frecuencias con GFN2-xTB

### ¿Por qué `--ohess` además de `--opt`?

La bandera `--ohess` calcula las **frecuencias vibracionales** en la
geometría optimizada en un solo paso y devuelve la energía libre de
Gibbs $G(298\,\text{K})$ directamente en el log:

$$G = E_\text{elec} + E_\text{ZPE} + H_\text{vib}(T) - TS_\text{vib}(T) + \ldots$$

Necesitamos $G$ — no solo $E$ — porque la distribución de Boltzmann
compara estabilidades termodinámicas, no energías electrónicas.

### Instrucciones

1. Ejecuta la celda. Tarda más que en P01 porque calcula frecuencias.
2. Observa: ¿todos los tautómeros convergieron?
3. **Frecuencias imaginarias**: si un tautómero tiene frecuencias
   negativas en el log, su geometría es un punto de silla, no un mínimo.
   La siguiente celda lo detecta automáticamente y lo excluye.


In [ ]:
# ── Paso 3a: optimización + frecuencias GFN2-xTB ──────────────
print('Ejecutando GFN2-xTB --opt tight --ohess para cada tautómero...')
print('(1-5 min dependiendo del número de tautómeros)\n')

resultados_xtb = []

for _, fila in df_geom[df_geom['embed_ok']].iterrows():
    etiqueta = fila['etiqueta']
    archivo  = os.path.basename(fila['etiqueta']) + '_FF.xyz'
    log_out  = os.path.join(directorio, f'{etiqueta}_xtb.out')
    t0 = time.time()

    if shutil.which('xtb'):
        result = subprocess.run(
            ['xtb', f'{etiqueta}_FF.xyz',
             '--opt', 'tight', '--ohess',
             '--chrg', '0', '--uhf', '0',
             '--gfn', '2', '--T', '298.15'],
            capture_output=True, text=True, cwd=directorio
        )
        log = result.stdout + result.stderr
    else:
        # Valor simulado para entornos sin xtb (Google Colab)
        E_sim = -17.52 - (hash(etiqueta) % 100) * 0.001
        G_sim = E_sim + 0.035
        log   = (f'  GEOMETRY OPTIMIZATION CONVERGED\n'
                 f'  TOTAL ENERGY      {E_sim:.8f} Eh\n'
                 f'  TOTAL FREE ENERGY {G_sim:.8f} Eh\n'
                 f'  RMS gradient    :   1.23E-05 Eh/a0\n')
        print(f'  \u26a0 xtb no disponible — valor simulado para {etiqueta}')

    t_total = time.time() - t0
    with open(log_out, 'w') as f:
        f.write(log)

    # Renombrar geometría final
    opt_src = os.path.join(directorio, 'xtbopt.xyz')
    opt_dst = os.path.join(directorio, f'{etiqueta}_opt.xyz')
    if os.path.exists(opt_src):
        shutil.move(opt_src, opt_dst)

    convergio  = 'GEOMETRY OPTIMIZATION CONVERGED' in log
    m_etot     = re.search(r'TOTAL ENERGY\s+([-\d.]+)', log)
    m_gibbs    = re.search(r'TOTAL FREE ENERGY\s+([-\d.]+)', log)
    m_freqs    = re.search(r'imaginary freq\.\s+(\d+)', log)
    n_imag     = int(m_freqs.group(1)) if m_freqs else 0

    resultados_xtb.append({
        'etiqueta'    : etiqueta,
        'smiles'      : fila['smiles'],
        'convergio'   : convergio,
        'E_total_Eh'  : float(m_etot.group(1))  if m_etot  else None,
        'G_298_Eh'    : float(m_gibbs.group(1)) if m_gibbs else None,
        'n_imaginarias': n_imag,
        't_xtb_s'     : t_total,
    })
    marca = '\u2713' if convergio and n_imag == 0 else '\u26a0'
    print(f'  {etiqueta}: {marca}  convergió={convergio}  freq_imag={n_imag}  t={t_total:.1f}s')

df_xtb = pd.DataFrame(resultados_xtb)
print(f'\nCompletados: {df_xtb["convergio"].sum()}/{len(df_xtb)} tautómeros')


In [ ]:
# ── Paso 3b: verificar frecuencias imaginarias ─────────────────
# Un tautómero con frecuencias imaginarias es un punto de silla,
# no un mínimo. Se excluye de la distribución de Boltzmann.

problemas = df_xtb[(~df_xtb['convergio']) | (df_xtb['n_imaginarias'] > 0)]

if len(problemas) > 0:
    print('\u26a0 Tautómeros con problemas:')
    for _, row in problemas.iterrows():
        print(f'  {row["etiqueta"]}: convergió={row["convergio"]}, '
              f'freq_imag={row["n_imaginarias"]}')
    print()
    print('Estos tautómeros quedan excluidos del análisis de Boltzmann.')
    print('Si quieres incluirlos, prueba --opt verytight y reoptimiza.')
else:
    print('\u2713 Todos los tautómeros convergieron sin frecuencias imaginarias.')
    print('  Cada geometría es un mínimo genuino en la PES de GFN2-xTB.')

df_validos = df_xtb[df_xtb['convergio'] & (df_xtb['n_imaginarias'] == 0)].copy()
print(f'\nTautómeros válidos para Boltzmann: {len(df_validos)}/{len(df_xtb)}')


---
## Paso 4 — Distribución de Boltzmann

### ¿Qué hace este paso?

Convertimos las energías libres $G_i$ en fracciones molares
de equilibrio:

$$p_i = \frac{e^{-G_i/RT}}{\displaystyle\sum_{j} e^{-G_j/RT}}$$

Esta expresión es la conexión central entre los números del cálculo
y la química observable: $p_i$ es la fracción de moléculas que
se encuentran en la forma $i$ en el equilibrio.

**Dato útil:** a 298 K, una diferencia de solo $\Delta G = 5.7$ kJ/mol
implica que la forma minoritaria representa ~10 % de la mezcla.
Una diferencia de 17 kJ/mol la reduce al 0.1 %.

### Instrucciones

1. Ejecuta la celda y observa la tabla de poblaciones.
2. Identifica el tautómero dominante y el segundo más estable.
3. Compara tu $\Delta G$ calculado con el valor experimental
   de Beak (1987): $\Delta G°_\text{gas} = -6.3$ kJ/mol a favor
   de la 2-piridona. ¿Tu resultado lo confirma?
4. Revisa tu respuesta a la Pregunta 2. ¿Cambias algo?


In [ ]:
# ── Paso 4: distribución de Boltzmann ─────────────────────────
df_boltz = df_validos.dropna(subset=['G_298_Eh']).copy()

if len(df_boltz) == 0:
    print('\u2717 No hay valores de G válidos.')
    print('  Verifica que xTB corrió con --ohess en el Paso 3.')
else:
    # ΔG relativo al mínimo (kJ/mol: 1 Eh = 2625.5 kJ/mol)
    G_min = df_boltz['G_298_Eh'].min()
    df_boltz['dG_kJ_mol'] = (df_boltz['G_298_Eh'] - G_min) * 2625.5

    # Pesos y poblaciones de Boltzmann
    df_boltz['peso']      = np.exp(-df_boltz['dG_kJ_mol'] / (R * T_K))
    df_boltz['poblacion'] = df_boltz['peso'] / df_boltz['peso'].sum()
    df_boltz['pct']       = df_boltz['poblacion'] * 100
    df_boltz = df_boltz.sort_values('dG_kJ_mol').reset_index(drop=True)

    print('=' * 65)
    print('  DISTRIBUCIÓN DE BOLTZMANN  —  2-piridona (gas, 298 K)')
    print('=' * 65)
    print(f'  {"Tautómero":22s}  {"G (Eh)":>13}  {"dG (kJ/mol)":>12}  {"pct":>7}')
    print('  ' + '-' * 60)
    for _, row in df_boltz.iterrows():
        marca = '  <- DOMINANTE' if row['dG_kJ_mol'] == 0 else ''
        print(f'  {row["etiqueta"]:22s}  {row["G_298_Eh"]:>13.8f}'
              f'  {row["dG_kJ_mol"]:>12.3f}  {row["pct"]:>7.2f}%{marca}')
    print()

    # Comparación con referencia experimental
    dG_T2 = df_boltz.loc[1, 'dG_kJ_mol'] if len(df_boltz) > 1 else None
    print('Comparación con referencia experimental (Beak 1987):')
    print(f'  dG experimental (fase gas) : -6.3 kJ/mol')
    if dG_T2 is not None:
        error = abs(dG_T2 - 6.3)
        marca = '\u2713' if error < 3.0 else '\u26a0'
        print(f'  dG calculado (GFN2-xTB)    :  {dG_T2:.2f} kJ/mol')
        print(f'  Error                      :  {error:.2f} kJ/mol  {marca}')
        print(f'  Criterio de aceptación     : < 3.0 kJ/mol')

    df_boltz.to_csv(f'{NOMBRE}_tautomeros_resultados.csv', index=False)
    print(f'\n\u2713 Guardado: {NOMBRE}_tautomeros_resultados.csv')


---
## Paso 5 — Visualización 3D de las formas tautómeras

### ¿Por qué antes de la validación numérica?

Antes de interpretar los números, hay que **ver** las geometrías.
Diez segundos de inspección visual detectan errores graves —
una molécula 'explotada', el H en la posición equivocada —
que ningún número te diría directamente.

### Instrucciones

1. Ejecuta la primera celda para ver el tautómero dominante.
2. **Rota la molécula** (clic + arrastrar). Responde en tu cuaderno:
   - ¿El H está sobre el N (forma ceto) o sobre el O (forma enol)?
   - ¿El anillo parece plano?
3. Ejecuta la comparación lado a lado: ¿puedes ver dónde migró el H?


In [ ]:
# ── Función de visualización 3D (misma que P01) ───────────────
def ver_xyz(archivo, titulo='', ancho=480, alto=360):
    if not PY3DMOL:
        print(f'py3Dmol no disponible. Abre {archivo} en Avogadro2.')
        return
    p = pathlib.Path(archivo)
    if not p.exists():
        print(f'Archivo no encontrado: {archivo}')
        return
    v = py3Dmol.view(width=ancho, height=alto)
    v.addModel(p.read_text(), 'xyz')
    v.setStyle({'stick': {'radius': 0.12}, 'sphere': {'scale': 0.22}})
    v.setBackgroundColor('white')
    if titulo:
        v.addLabel(titulo, {'position': {'x':0,'y':2,'z':0},
                             'fontColor':'black','fontSize':13})
    v.zoomTo(); v.show()

# Visualizar el tautómero dominante
if len(df_boltz) > 0:
    t1 = df_boltz.iloc[0]
    t1_xyz = os.path.join(directorio, f'{t1["etiqueta"]}_opt.xyz')
    print(f'Tautómero dominante: {t1["etiqueta"]}  ({t1["pct"]:.1f}%)')
    print(f'SMILES: {t1["smiles"]}')
    ver_xyz(t1_xyz, titulo=f'{t1["etiqueta"]} ({t1["pct"]:.1f}%)')


In [ ]:
# ── Comparación lado a lado — los dos tautómeros más estables ──
if PY3DMOL and len(df_boltz) >= 2:
    t1, t2 = df_boltz.iloc[0], df_boltz.iloc[1]
    xyz1 = pathlib.Path(directorio) / f'{t1["etiqueta"]}_opt.xyz'
    xyz2 = pathlib.Path(directorio) / f'{t2["etiqueta"]}_opt.xyz'
    if xyz1.exists() and xyz2.exists():
        v = py3Dmol.view(width=800, height=360, viewergrid=(1,2))
        v.addModel(xyz1.read_text(), 'xyz', viewer=(0,0))
        v.addModel(xyz2.read_text(), 'xyz', viewer=(0,1))
        for g in [(0,0),(0,1)]:
            v.setStyle({'stick':{'radius':0.12},'sphere':{'scale':0.22}}, viewer=g)
        v.addLabel(f'{t1["etiqueta"]} — {t1["pct"]:.1f}%',
                   {'position':{'x':0,'y':2,'z':0},'fontColor':'steelblue'}, viewer=(0,0))
        v.addLabel(f'{t2["etiqueta"]} — {t2["pct"]:.1f}%',
                   {'position':{'x':0,'y':2,'z':0},'fontColor':'darkorange'}, viewer=(0,1))
        v.setBackgroundColor('white'); v.zoomTo(); v.show()
        print('Izquierda: tautómero más estable')
        print('Derecha  : segundo más estable')
        print()
        print('¿Puedes localizar el H migratorio en cada forma?')
        print('¿El enlace C=O es más corto o más largo en la forma ceto?')
else:
    print('Abre los archivos _opt.xyz en Avogadro2 para comparar.')


---
## Paso 6 — Validación geométrica del tautómero dominante

### ¿Qué verificamos?

Para confirmar que la geometría optimizada realmente corresponde
a la forma tautómera esperada, comparamos distancias características:

| Enlace | Forma ceto (2-piridona) | Forma enol (2-OH-piridina) |
|:--|:--|:--|
| C=O (carbonilo) | ~1.22 Å | — |
| C–OH (fenólico) | — | ~1.34 Å |
| O–H | — | ~0.97 Å |
| N–H (lactama) | ~1.01 Å | — |

### Instrucciones

1. Ejecuta la celda.
2. Compara las distancias calculadas con los valores de la tabla.
3. ¿Confirman los parámetros geométricos la identidad del tautómero dominante?
4. ¿Es consistente con tu predicción en la Pregunta 2?


In [ ]:
# ── Funciones de geometría (reutilizadas de P01) ──────────────
def leer_xyz(archivo):
    with open(archivo) as f:
        lineas = f.readlines()
    n = int(lineas[0])
    simbolos, coords = [], []
    for l in lineas[2:2+n]:
        p = l.split()
        simbolos.append(p[0])
        coords.append([float(x) for x in p[1:4]])
    return simbolos, np.array(coords)

def dist_enlace(coords, i, j):
    return float(np.linalg.norm(coords[i] - coords[j]))

print('Funciones de geometría cargadas.')


In [ ]:
# ── Validación geométrica del tautómero dominante ─────────────
if len(df_boltz) > 0:
    t1_label = df_boltz.iloc[0]['etiqueta']
    t1_xyz   = os.path.join(directorio, f'{t1_label}_opt.xyz')

    if pathlib.Path(t1_xyz).exists():
        simbolos, coords = leer_xyz(t1_xyz)
        print(f'Tautómero: {t1_label}')
        print(f'Composición: { {e: simbolos.count(e) for e in sorted(set(simbolos))} }')
        print()

        mol_ref   = Chem.MolFromSmiles(df_boltz.iloc[0]['smiles'])
        mol_ref_h = Chem.AddHs(mol_ref)
        p = AllChem.ETKDGv3(); p.randomSeed = 42
        AllChem.EmbedMolecule(mol_ref_h, p)

        pat_CO  = Chem.MolFromSmarts('[C:1]=[O:2]')
        pat_COH = Chem.MolFromSmarts('[c:1]-[OH:2]')
        pat_NH  = Chem.MolFromSmarts('[N:1]-[H:2]')

        print(f'  {"Enlace":<22}  {"Calculado (Å)":>14}  {"Referencia (Å)":>16}')
        print('  ' + '-' * 56)

        for pat, label, ref in [
            (pat_CO,  'C=O (carbonilo)',     '~1.22 (lactama)'),
            (pat_COH, 'C-OH (enol)',         '~1.34 (fenólico)'),
            (pat_NH,  'N-H (lactama)',        '~1.01 (amida)'),
        ]:
            matches = mol_ref_h.GetSubstructMatches(pat)
            if matches:
                i, j = matches[0][0], matches[0][1]
                d = dist_enlace(coords, i, j)
                print(f'  {label:<22}  {d:>14.4f}  {ref:>16}')
            else:
                print(f'  {label:<22}  {"no encontrado":>14}  {ref:>16}')

        print()
        print('¿Las distancias confirman que es la forma ceto o la enol?')
        print('¿Es consistente con tu predicción en la Pregunta 2?')
    else:
        print(f'No se encontró {t1_xyz}. Verifica que el Paso 3 completó.')


---
## 🌳 Parte 2 · El bosque — 40 moléculas

Ahora que dominas el pipeline con la 2-piridona, analizamos
un dataset de 40 moléculas con tautomería documentada:
bases nucleicas, fármacos, heterociclos simples y compuestos
con tautomería ceto-enol.

El objetivo es responder preguntas que el cálculo individual
no puede contestar: ¿qué familias tienen mayor incertidumbre
tautómera? ¿Dónde falla GFN2-xTB respecto a datos experimentales?

### Estructura del dataset

| Columna | Tipo | Descripción |
|:--|:--|:--|
| `id` | str | nombre del compuesto |
| `smiles_entrada` | str | SMILES de partida |
| `familia` | str | categoría estructural |
| `n_tautomeros` | int | número enumerado por RDKit |
| `smiles_T1` | str | SMILES del más estable |
| `dG_T2_kJ` | float | ΔG del segundo más estable (kJ/mol) |
| `pop_T1_pct` | float | población del dominante (%) |
| `dG_ref_kJ` | float | valor experimental si existe |

### Las 4 familias estructurales

- **Bases nucleicas** (8): adenina, guanina, citosina, uracilo y análogos
- **Fármacos** (12): omeprazol, pirimetamina, imatinib (fragmento), entre otros
- **Heterociclos simples** (10): 4-piridona, imidazol, triazoles, tetrazoles
- **Ceto-enol no obvia** (10): β-cetoésteres, malonaldehído, acetilacetona


In [ ]:
# ── Cargar el dataset del bosque ──────────────────────────────
try:
    df = pd.read_csv('p02_bosque_resultados.csv')
    print(f'\u2713 Dataset cargado: {len(df)} moléculas')
    print()
    print('Primeras 3 filas:')
    display(df.head(3))
except FileNotFoundError:
    print('\u2717 No se encontró p02_bosque_resultados.csv')
    print()
    print('Opciones:')
    print('  1. Descárgalo del repositorio del curso:')
    print('     https://github.com/qcmanual/del-orbital-al-espacio-quimico')
    print('     ruta: data/p02_bosque_resultados.csv')
    print('  2. Genera el dataset ejecutando:')
    print('     python scripts/p02_expansion.py  (~2 h en laptop)')
    raise


In [ ]:
# ── Integrar la semilla al dataset ────────────────────────────
# Los valores se leen automáticamente de los cálculos anteriores.
# Si no terminaron, deja None — el análisis seguirá funcionando.

semilla = {
    'id'             : '2-piridona',
    'smiles_entrada' : SMILES_ENTRADA,
    'familia'        : 'heterociclo_ceto_enol',
    'n_tautomeros'   : n_tau if 'n_tau' in dir() else None,
    'smiles_T1'      : df_boltz.iloc[0]['smiles'] if len(df_boltz) > 0 else None,
    'dG_T1_kJ'       : 0.0,
    'dG_T2_kJ'       : df_boltz.iloc[1]['dG_kJ_mol'] if len(df_boltz) > 1 else None,
    'pop_T1_pct'     : df_boltz.iloc[0]['pct'] if len(df_boltz) > 0 else None,
    'pop_T2_pct'     : df_boltz.iloc[1]['pct'] if len(df_boltz) > 1 else None,
    'dG_ref_kJ'      : -6.3,
    'referencia'     : 'beak1987',
}

if '2-piridona' not in df['id'].values:
    df = pd.concat([df, pd.DataFrame([semilla])], ignore_index=True)
    print('\u2713 2-piridona añadida al dataset.')
else:
    for col, val in semilla.items():
        if val is not None:
            df.loc[df['id'] == '2-piridona', col] = val
    print('\u2713 Fila de 2-piridona actualizada.')

df.to_csv('p02_dataset_final.csv', index=False)
print(f'  Dataset final: {len(df)} moléculas  ->  p02_dataset_final.csv')
print()
print('Moléculas por familia:')
print(df['familia'].value_counts().to_string())


---
## Análisis del bosque

Cada gráfica responde una pregunta química concreta.
**Lee la pregunta antes de ejecutar**, formula tu predicción
y luego compara con el resultado.


In [ ]:
# ── Gráfica 1: Distribución de poblaciones del bosque ─────────
# Pregunta: ¿qué fracción de las 40 moléculas tiene un tautómero
# dominante claro (p1 > 95%)? ¿Cuántas muestran incertidumbre
# tautómera significativa (p1 < 80%)?
# Predice antes de ejecutar: ¿cuál familia esperas que tenga
# mayor incertidumbre — bases nucleicas o heterociclos simples?

df_plot = df.dropna(subset=['pop_T1_pct']).sort_values('pop_T1_pct').reset_index(drop=True)

colores = ['#0F2747' if p > 95 else
           '#2A6F97' if p > 80 else
           '#AABBD0'
           for p in df_plot['pop_T1_pct']]

fig, ax = plt.subplots(figsize=(11, 5))
bars = ax.bar(range(len(df_plot)), df_plot['pop_T1_pct'],
              color=colores, edgecolor='white', linewidth=0.5)

# Destacar la semilla en rojo
idx_s = df_plot.index[df_plot['id'] == '2-piridona'].tolist()
if idx_s:
    bars[idx_s[0]].set_edgecolor('red'); bars[idx_s[0]].set_linewidth(2)
    ax.annotate('2-piridona (semilla)',
                xy=(idx_s[0], df_plot.loc[idx_s[0], 'pop_T1_pct']),
                xytext=(idx_s[0]+2, 70),
                arrowprops=dict(arrowstyle='->', color='red'),
                color='red', fontsize=9)

ax.axhline(95, color='#444', linestyle='--', linewidth=0.8, label='95 %')
ax.axhline(80, color='#888', linestyle=':', linewidth=0.8, label='80 %')
ax.set_xlabel('Molécula (ordenada por población dominante)', fontsize=11)
ax.set_ylabel('Población del tautómero más estable (%)', fontsize=11)
ax.set_title('Incertidumbre tautómera en el bosque de 40 moléculas', fontsize=12)
ax.legend(fontsize=9); ax.set_ylim(0, 107)
plt.tight_layout()
plt.savefig('p02_poblaciones_barra.pdf', bbox_inches='tight')
plt.show()

n_dom = (df_plot['pop_T1_pct'] > 95).sum()
n_inc = (df_plot['pop_T1_pct'] < 80).sum()
print(f'Tautómero dominante claro (p1 > 95 %): {n_dom}/{len(df_plot)} moléculas')
print(f'Incertidumbre significativa (p1 < 80 %): {n_inc}/{len(df_plot)} moléculas')
print()
print('¿La distribución confirma tu predicción?')
print('¿Cuál es la molécula con mayor incertidumbre tautómera?')


In [ ]:
# ── Gráfica 2: dG_T2 vs p1, coloreada por familia ─────────────
# Pregunta: ¿existe una familia estructural que concentre
# los casos de mayor incertidumbre tautómera?
# ¿Qué característica química predice mejor la incertidumbre?

df_g2 = df.dropna(subset=['dG_T2_kJ', 'pop_T1_pct'])
familias = sorted(df_g2['familia'].unique())
colmap   = plt.cm.tab10(np.linspace(0, 1, len(familias)))

fig, ax = plt.subplots(figsize=(7, 5))
for fam, color in zip(familias, colmap):
    sub = df_g2[df_g2['familia'] == fam]
    ax.scatter(sub['dG_T2_kJ'], sub['pop_T1_pct'], label=fam,
               color=color, s=65, alpha=0.85, edgecolors='k', linewidths=0.3)

# Destacar semilla
sem = df_g2[df_g2['id'] == '2-piridona']
if len(sem) > 0:
    ax.scatter(sem['dG_T2_kJ'], sem['pop_T1_pct'], color='red',
               s=140, zorder=5, marker='*', label='2-piridona (semilla)')

ax.set_xlabel(r'$\Delta G_{T_2}$ (kJ/mol)', fontsize=12)
ax.set_ylabel(r'Población $p_1$ (%)', fontsize=12)
ax.set_title('Incertidumbre tautómera por familia estructural', fontsize=12)
ax.legend(fontsize=8, loc='lower right')
plt.tight_layout()
plt.savefig('p02_dG_vs_pop.pdf', bbox_inches='tight')
plt.show()

print('¿Qué familia tiene los puntos con menor dG_T2?')
print('¿Tiene sentido desde el punto de vista de la química orgánica?')


In [ ]:
# ── Gráfica 3: Validación — calculado vs experimental ──────────
# Pregunta: ¿dónde falla GFN2-xTB vs datos experimentales?
# ¿El error es sistemático (siempre en la misma dirección)
# o aleatorio? ¿A qué se debe?

df_val = df.dropna(subset=['dG_ref_kJ', 'dG_T2_kJ'])

if len(df_val) < 2:
    print('Pocos datos con referencia experimental disponibles.')
    print('La gráfica se generará cuando el bosque tenga más referencias.')
else:
    fig, ax = plt.subplots(figsize=(5, 5))
    ax.scatter(df_val['dG_ref_kJ'], df_val['dG_T2_kJ'],
               edgecolors='k', linewidths=0.4, s=70, zorder=3)

    # Etiquetar la semilla
    for _, row in df_val[df_val['id'] == '2-piridona'].iterrows():
        ax.annotate('2-piridona',
                    xy=(row['dG_ref_kJ'], row['dG_T2_kJ']),
                    xytext=(row['dG_ref_kJ']+1, row['dG_T2_kJ']-2),
                    fontsize=8, color='red')

    vals = pd.concat([df_val['dG_ref_kJ'], df_val['dG_T2_kJ']])
    lim  = max(abs(vals.min()), abs(vals.max())) + 3
    ax.plot([-lim, lim], [-lim, lim], 'k--', linewidth=0.8, label='Paridad perfecta')
    ax.set_xlim(-lim, lim); ax.set_ylim(-lim, lim)
    ax.set_xlabel(r'$\Delta G_{\mathrm{exp}}$ (kJ/mol)', fontsize=12)
    ax.set_ylabel(r'$\Delta G_{\mathrm{calc}}$ (kJ/mol)', fontsize=12)
    ax.set_title('GFN2-xTB (gas) vs experimental', fontsize=12)
    ax.legend(fontsize=9)

    rmse = np.sqrt(((df_val['dG_T2_kJ'] - df_val['dG_ref_kJ'])**2).mean())
    ax.text(0.05, 0.92, f'RMSE = {rmse:.2f} kJ/mol',
            transform=ax.transAxes, fontsize=10)
    plt.tight_layout()
    plt.savefig('p02_paridad.pdf', bbox_inches='tight')
    plt.show()

    print(f'RMSE total: {rmse:.2f} kJ/mol')
    print()
    print('¿Los puntos están sistemáticamente por encima o por debajo')
    print('de la línea de paridad? ¿Qué implica sobre las limitaciones')
    print('de calcular en vacío para predecir equilibrios en solución?')


---
## 🔄 Revisión de hipótesis iniciales

Vuelve a las respuestas que escribiste al principio.


In [ ]:
# ── Revisión de respuestas previas ────────────────────────────
print('=' * 58)
print('  TUS RESPUESTAS PREVIAS')
print('=' * 58)
print()
print('Pregunta 1 (tautómeros de la guanina):')
print(respuesta_1)
print()
print('Pregunta 2 (forma ceto vs enol de la 2-piridona):')
print(respuesta_2)
print()
print('Pregunta 3 (limitaciones del protocolo en vacío):')
print(respuesta_3)
print()
print('-' * 58)
print('Después de la práctica, responde en la celda siguiente:')
print('  · ¿Tu predicción sobre la forma dominante fue correcta?')
print('  · ¿Qué fue lo más sorprendente del bosque?')
print('  · ¿Qué pregunta nueva te generó la práctica?')


In [ ]:
# ── Escribe tu reflexión final ─────────────────────────────────
reflexion_final = (
    '¿Tu predicción sobre la forma dominante fue correcta?:\n'
    '\n'
    '¿Qué fue lo más sorprendente del análisis del bosque?:\n'
    '\n'
    '¿Qué limitación del protocolo observaste claramente?:\n'
    '\n'
    '¿Qué pregunta nueva te generó esta práctica?:\n'
)
print(reflexion_final)


---
## 📦 Entregables

| Etiqueta | Archivo | Descripción |
|:--|:--|:--|
| D1 | `2-piridona_tautomeros_resultados.csv` | G, ΔG y poblaciones de todos los tautómeros |
| D2 | `2-piridona_T01_xtb.out` y `T02_xtb.out` | Logs de xTB de las dos formas más estables |
| D3 | `p02_dataset_final.csv` | Bosque completo con semilla integrada |
| F1 | `p02_poblaciones_barra.pdf` | Distribución de p₁ para el bosque |
| F2 | `p02_dG_vs_pop.pdf` | ΔG_T2 vs p₁ coloreada por familia |
| F3 | `p02_paridad.pdf` | Calculado vs experimental |
| T1 | Reporte (≤ 2 páginas) | Tabla validación + F1 y F2 comentadas + preguntas de discusión |


In [ ]:
# ── Checklist automático de entregables ───────────────────────
entregables = {
    f'{NOMBRE}_tautomeros_resultados.csv'  : 'D1 — Resultados semilla',
    f'{directorio}/{NOMBRE}_T01_xtb.out'   : 'D2 — Log xTB T01',
    f'{directorio}/{NOMBRE}_T02_xtb.out'   : 'D2 — Log xTB T02',
    'p02_dataset_final.csv'                : 'D3 — Dataset final',
    'p02_poblaciones_barra.pdf'            : 'F1 — Barras de población',
    'p02_dG_vs_pop.pdf'                    : 'F2 — ΔG vs p₁',
    'p02_paridad.pdf'                      : 'F3 — Paridad calc/exp',
}

print('=' * 55)
print('  CHECKLIST DE ENTREGABLES')
print('=' * 55)
todos = True
for archivo, desc in entregables.items():
    existe = pathlib.Path(archivo).exists()
    if existe:
        size = pathlib.Path(archivo).stat().st_size
        print(f'  \u2713  {desc:<38s} ({size:,} bytes)')
    else:
        print(f'  \u2717  {desc:<38s} <- FALTA')
        todos = False

print()
if todos:
    print('\u2713 Todos los entregables están listos.')
else:
    print('\u26a0 Algunos entregables faltan. Revisa las celdas marcadas con \u2717.')


---
## ❓ Preguntas de discusión

Responde en el reporte escrito (≤ 150 palabras por pregunta).
Puedes usar esta celda para esbozar ideas antes de redactar.


In [ ]:
preguntas = {
    1: ('(Comprensión) ¿Cuántos tautómeros encontró TautomerEnumerator '
        'para la 2-piridona? ¿Son todos químicamente razonables? '
        'Si alguno no lo es, explica por qué el algoritmo lo genera.'),

    2: ('(Comprensión) Examina el log de xTB del tautómero dominante. '
        'Identifica la línea que reporta G(298 K). '
        '¿Qué diferencia hay entre E_total y G(298 K)? '
        '¿Qué contribuciones energéticas suma xTB para obtener G?'),

    3: ('(Análisis) Usa la ecuación de Boltzmann para calcular manualmente '
        'qué ΔG sería necesario para que el tautómero minoritario tuviera '
        'exactamente 1 % de población. ¿A qué temperatura podría ser '
        'observable con una técnica espectroscópica estándar?'),

    4: ('(Análisis) Del bosque, identifica la molécula con la distribución '
        'más plana (menor dG_T2, mayor incertidumbre tautómera). '
        '¿Qué consecuencia tendría para un cálculo de docking si '
        'se usara el tautómero incorrecto como punto de partida?'),

    5: ('(Metodológico) El protocolo usa energías en vacío. '
        '¿En qué dirección esperarías que se desplazara el equilibrio '
        '2-piridona <-> 2-hidroxipiridina al pasar de vacío a agua? '
        '¿A ciclohexano? Justifica con argumentos de polaridad y '
        'puentes de hidrógeno.'),
}

for n, q in preguntas.items():
    print(f'Pregunta {n}:')
    print(q)
    print()


---
## 🔗 Conexión con el resto del manual

```{admonition} Guarda estos archivos — los usarás en prácticas futuras
:class: tip

- `2-piridona_T01_opt.xyz` → input de la **Práctica 4**
  (el tautómero dominante es el punto de partida correcto para DFT)
- `p02_dataset_final.csv` → referencia para la **Práctica 30**
  (efecto del solvente PCM/SMD sobre equilibrios tautómeros)
- El pipeline de Boltzmann se reutiliza en la **Práctica 3**
  y en los **Bloques 17–18** (diseño molecular)
```

| Habilidad aprendida | Herramienta | Vuelve a aparecer en... |
|:--|:--|:--|
| Enumeración de tautómeros | `TautomerEnumerator` | P47–P50 (normalización) |
| Optimización + frecuencias | `xtb --ohess` | P10–P12 (termoquímica) |
| Distribución de Boltzmann | NumPy | P03, P30, P34 |
| Gráfico de paridad calc/exp | matplotlib | P04–P18 (validación) |
| Análisis por familia estructural | pandas groupby | P47–P55 |
